# Vehicle Sales Data Cleaning

This notebook applies cleaning decisions based on findings from the initial data audit.

The raw dataset will remain unchanged. Cleaning will be performed on a separate copy of the data.

In [1]:
import pandas as pd

In [2]:
cars = pd.read_csv("../data/raw/car_prices.csv")

In [3]:
clean = cars.copy()

In [4]:
cars.shape

(558837, 16)

In [5]:
clean.shape

(558837, 16)

In [7]:
malformed_rows = (
    clean["transmission"].isin(["Sedan", "sedan"])
    & clean["color"].astype(str).str.fullmatch(r"\d+")
)

In [8]:
malformed_rows.sum()

np.int64(26)

In [9]:
removed_malformed = clean[malformed_rows].copy()

In [10]:
clean = clean[~malformed_rows].copy()

In [11]:
clean.shape

(558811, 16)

## 1. Remove Malformed Records

The audit identified 26 Volkswagen Jetta records with values shifted across multiple columns.

Rather than attempting to reconstruct these records using assumptions, they are removed from the cleaned dataset. The original raw dataset remains unchanged, and the removed records are retained separately for traceability.

## 2. Normalize Vehicle Condition Scores

The raw `condition` column uses two formats:

- Whole-number grades such as `1`, `2`, `3`, `4`, and `5`
- Decimal grades stored without the decimal point, such as `21` for `2.1` and `49` for `4.9`

A new `condition_clean` column will be created so the original condition values remain available for comparison.

In [12]:
clean["condition_clean"] = clean["condition"].copy()

In [13]:
condition_needs_conversion = clean["condition_clean"] > 5

In [16]:
clean.loc[condition_needs_conversion, "condition_clean"] = (
    clean.loc[condition_needs_conversion, "condition_clean"] / 10
)

In [17]:
sorted(clean["condition_clean"].dropna().unique())

[np.float64(1.0),
 np.float64(1.1),
 np.float64(1.2),
 np.float64(1.3),
 np.float64(1.4),
 np.float64(1.5),
 np.float64(1.6),
 np.float64(1.7),
 np.float64(1.8),
 np.float64(1.9),
 np.float64(2.0),
 np.float64(2.1),
 np.float64(2.2),
 np.float64(2.3),
 np.float64(2.4),
 np.float64(2.5),
 np.float64(2.6),
 np.float64(2.7),
 np.float64(2.8),
 np.float64(2.9),
 np.float64(3.0),
 np.float64(3.1),
 np.float64(3.2),
 np.float64(3.3),
 np.float64(3.4),
 np.float64(3.5),
 np.float64(3.6),
 np.float64(3.7),
 np.float64(3.8),
 np.float64(3.9),
 np.float64(4.0),
 np.float64(4.1),
 np.float64(4.2),
 np.float64(4.3),
 np.float64(4.4),
 np.float64(4.5),
 np.float64(4.6),
 np.float64(4.7),
 np.float64(4.8),
 np.float64(4.9),
 np.float64(5.0)]

In [18]:
clean[["condition_clean","condition"]].head(20)

,condition_clean,condition
0,5.0,5.0
1,5.0,5.0
2,4.5,45.0
3,4.1,41.0
4,4.3,43.0
5,1.0,1.0
6,3.4,34.0
7,2.0,2.0
8,4.2,42.0
9,3.0,3.0


In [20]:
make_mapping = (
    clean.dropna(subset=["make"])
    .assign(make_key=lambda df: df["make"].str.lower())
    .groupby("make_key")["make"]
    .agg(lambda x: x.value_counts().idxmax())
)

In [21]:
clean["make_clean"] = clean["make"].str.lower().map(make_mapping)

In [29]:
clean["make_clean"].value_counts().head(40)

make_clean
Ford             93997
Chevrolet        60587
Nissan           54017
Toyota           39966
Dodge            30955
Honda            27351
Hyundai          21836
BMW              20793
Kia              18084
Chrysler         17485
Mercedes-Benz    17141
Jeep             15483
Infiniti         15305
Volkswagen       12579
Lexus            11980
GMC              10638
Mazda             8508
Cadillac          7629
Acura             5926
Audi              5877
Lincoln           5786
Buick             5121
Subaru            5103
Ram               4574
Pontiac           4524
Mitsubishi        4257
Volvo             3788
MINI              3224
Saturn            2841
Mercury           2023
Land Rover        1864
Scion             1687
Jaguar            1420
Porsche           1402
Suzuki            1078
FIAT               865
HUMMER             805
Saab               484
smart              396
Oldsmobile         384
Name: count, dtype: int64

In [24]:
clean["make_clean"].nunique(dropna=True)

66

In [25]:
body_mapping = (
    clean.dropna(subset=["body"])
    .assign(body_key=lambda df: df["body"].str.lower())
    .groupby("body_key")["body"]
    .agg(lambda x: x.value_counts().idxmax())
)

In [26]:
clean["body_clean"] = clean["body"].str.lower().map(body_mapping)

In [27]:
clean["body"].value_counts().head(40)

body
Sedan            199437
SUV              119292
sedan             41906
suv               24552
Hatchback         21380
Minivan           21363
Coupe             14602
Wagon             13630
Crew Cab          13280
Convertible        8652
SuperCrew          7423
G Sedan            5999
hatchback          4857
SuperCab           4449
minivan            4166
Regular Cab        4067
Van                3958
Extended Cab       3824
Quad Cab           3436
coupe              3150
crew cab           3114
wagon              2499
convertible        1824
supercrew          1610
E-Series Van       1455
g sedan            1418
G Coupe            1263
Double Cab         1251
supercab            862
regular cab         783
extended cab        683
quad cab            659
van                 570
CrewMax Cab         445
King Cab            436
e-series van        368
double cab          350
g coupe             330
G Convertible       249
Genesis Coupe       232
Name: count, dtype: int64

In [28]:
clean["body_clean"].nunique(dropna=True)

45

In [30]:
clean.loc[
    clean["make"].notna() & (clean["make"] != clean["make_clean"]),
    ["make", "make_clean"]
].drop_duplicates().head(30)

,make,make_clean
468,lincoln,Lincoln
743,bmw,BMW
977,ford,Ford
1563,mitsubishi,Mitsubishi
2525,chevrolet,Chevrolet
2615,chrysler,Chrysler
2719,pontiac,Pontiac
3346,cadillac,Cadillac
3470,land rover,Land Rover
3519,mazda,Mazda


In [31]:
clean.loc[
    clean["body"].notna() & (clean["body"] != clean["body_clean"]),
    ["body", "body_clean"]
].drop_duplicates().head(30)

,body,body_clean
2596,sedan,Sedan
10491,convertible,Convertible
234616,minivan,Minivan
366047,suv,SUV
419190,van,Van
463408,g sedan,G Sedan
463411,g coupe,G Coupe
463449,hatchback,Hatchback
463454,king cab,King Cab
463603,supercrew,SuperCrew


In [32]:
clean["transmission"].value_counts(dropna=False)

transmission
automatic    475915
NaN           65352
manual        17544
Name: count, dtype: int64

In [33]:
clean["transmission_clean"] = clean["transmission"].copy()

In [34]:
clean["transmission_clean"].value_counts(dropna=False)

transmission_clean
automatic    475915
NaN           65352
manual        17544
Name: count, dtype: int64

In [35]:
clean["transmission"].value_counts(dropna=False)

transmission
automatic    475915
NaN           65352
manual        17544
Name: count, dtype: int64

## 4. Transmission Check

After removing the 26 malformed records, the `transmission` column contains only `automatic`, `manual`, and missing values.

No additional transmission cleaning is required.

In [41]:
clean["color"].value_counts(dropna=False).head(30)

color
black        110970
white        106673
silver        83389
gray          82857
blue          51139
red           43569
—             24685
green         11382
gold          11342
beige          9222
burgundy       8972
brown          6717
orange         2078
purple         1561
off-white      1449
yellow         1285
NaN             749
charcoal        479
turquoise       236
pink             42
lime             15
Name: count, dtype: int64

In [46]:
clean[clean["color"] == "—"][
    ["year", "make", "model", "body", "transmission", "color", "interior", "seller"]
    ].head(20)

,year,make,model,body,transmission,color,interior,seller
134,2013,Infiniti,G Sedan,G Sedan,automatic,—,black,nissan infiniti lt
154,2013,Infiniti,G Coupe,G Coupe,automatic,—,black,nissan infiniti lt
185,2013,Infiniti,G Sedan,G Sedan,automatic,—,beige,nissan infiniti lt
282,2013,Infiniti,G Sedan,G Sedan,automatic,—,black,nissan infiniti lt
322,2013,Infiniti,G Coupe,G Coupe,automatic,—,black,nissan infiniti lt
341,2013,Hyundai,Sonata,Sedan,automatic,—,beige,hyundai motor finance
350,2013,Infiniti,G Sedan,G Sedan,automatic,—,beige,nissan infiniti lt
362,2013,Mercedes-Benz,C-Class,Coupe,automatic,—,black,mercedes-benz financial services
376,2013,Infiniti,M,Sedan,automatic,—,black,nissan north america inc.
443,2013,Infiniti,G Coupe,G Coupe,automatic,—,black,nissan infiniti lt


In [48]:
clean.loc[clean["color"] == "—", "make"].value_counts().head(15)

make
Ford             9980
Nissan           3296
Mercedes-Benz    1709
Volkswagen       1119
Chevrolet         981
Toyota            975
Infiniti          898
Lincoln           854
Dodge             621
BMW               547
Lexus             415
Hyundai           363
Honda             320
Chrysler          319
Jeep              292
Name: count, dtype: int64

In [49]:
clean.loc[clean["color"] == "—", "year"].value_counts().head(15)

year
2012    6360
2014    6299
2013    6156
2011    2356
2010     700
2008     492
2007     415
2015     402
2009     376
2006     321
2005     254
2004     192
2003     109
2002      77
2000      51
Name: count, dtype: int64

In [51]:
clean.loc[clean["color"] == "—", "seller"].value_counts().head(15)

seller
ford motor credit company llc            2631
avis budget group                        1728
nissan-infiniti lt                       1547
mercedes-benz financial services         1285
ford motor credit company llc pd         1029
r hollenshead auto sales inc              846
lease plan usa                            830
vw credit                                 768
nissan motor acceptance corporation       764
wheels exchange llc                       747
nissan infiniti lt                        724
ford motor credit company                 665
ars/avis budget group                     651
ge fleet services for itself/servicer     605
ford-lincoln dealer program               491
Name: count, dtype: int64

In [52]:
clean["color_clean"] = clean["color"].replace("—", pd.NA)

In [55]:
clean["color_clean"].value_counts(dropna=False)

color_clean
black        110970
white        106673
silver        83389
gray          82857
blue          51139
red           43569
NaN           25434
green         11382
gold          11342
beige          9222
burgundy       8972
brown          6717
orange         2078
purple         1561
off-white      1449
yellow         1285
charcoal        479
turquoise       236
pink             42
lime             15
Name: count, dtype: int64

## 5. Exterior Color Investigation

The `color` column contains a large number of records represented by an em dash (`—`).

Inspection of these rows shows that the surrounding vehicle information appears normally aligned, unlike the previously removed malformed Volkswagen Jetta records.

This suggests that `—` may represent an unrecorded or unavailable exterior color rather than a corrupted row.

No replacement has been made yet. The distribution of these records across makes, sellers, and model years will be checked before deciding how to handle them.